In [68]:
import numpy as np
import pandas as pd
import matplotlib as mb

In [147]:
class Node:
    thresholds=[]
    maxDept=10
    def __init__(self, left=None, right=None, value = None, dept=None, threshold = None):
        self.threshold=threshold
        self.left = left
        self.right = right
        
    def fillThreshold(self,X,Y,method):
        m = X.shape[1]
        Node.thresholds = [None] * m
        for i in range(m):
            bestValue = best_Value(X,Y,i,method)
            Node.thresholds[i]=bestValue

    def createNode(self, left=None, right=None, threshold = None):
        return Node(left=None, right=None, threshold = None)

In [70]:
def compute_impurity(Y):
    impurity=.0
    if(len(Y) == 0) : return 0
    if(len(np.unique(Y)) == 1) : return 0

    p1= len(Y[Y==1]) / len(Y)
    p2=1-p1
    impurity = np.log2(p1) * (-p1) - np.log2(p2) * p2
    return impurity

In [71]:
def quintile_threshold(X):
    X_sorted= np.sort(X)
    quintiles = np.linspace(0,100,10)[1:-1]
    values= np.ones(len(quintiles))
    a=0
    for i in quintiles:
        values[a] = np.round(np.percentile(X_sorted,i),2)
        a = a+1
    return values

In [72]:
def unique_threshold(X):
    X_sorted= np.sort(X)
    X_uniqued=np.unique(X_sorted)
    thresholds=(X_uniqued[1:] + X_uniqued[:-1])/2
    return thresholds

In [87]:
def best_Value(X,Y,feature_no,method):
    values = method(X[:,feature_no])
    maxGain=0
    bestValue=-1
    for i in values:
        if(info_gain(X,Y,feature_no,i)[0] > maxGain):
            maxGain=info_gain(X,Y,feature_no,i)[0]
            bestValue=i
    return bestValue

In [88]:
def one_hot_encoding(X,Y,method):
    for feature in range(X.shape[1]):
        threshold=best_Value(X,Y,feature,method)
        X[:,feature] = (X[:,feature] > threshold)

In [89]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X = data.data
Y = data.target

In [122]:
len(pd.DataFrame(Y))

569

In [92]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size = 0.2,random_state=42)

In [94]:
for i in range(5):
    print(Node.thresholds[i])

15.025
18.46
98.43
696.25
0.08946499999999999


In [95]:
def info_gain(X, Y, feature_no, threshold):
    if len(Y) <= 1: return 0
    base_impurity = compute_impurity(Y)
    feature_values = X[:, feature_no]

    mask_left = feature_values > threshold
    mask_right = feature_values < threshold

    X_left = X[mask_left]
    X_right = X[mask_right]
    Y_left = Y[mask_left]
    Y_right = Y[mask_right]

    p_left = compute_impurity(Y_left)
    p_right = compute_impurity(Y_right)

    w_left = len(Y_left) / len(Y)
    w_right = len(Y_right) / len(Y)

    return [base_impurity - (p_left * w_left + p_right * w_right), X_left, X_right, Y_left, Y_right]

In [150]:
def splitTree(node,X,Y,dept):
    
    if(len(Y) < 10): return None 
    if(compute_impurity(Y) == 0): return node
    if(dept >= Node.maxDept): return node

    
    threshold = 0
    bestGain=[0]
    for i in range (len(Node.thresholds)):
        gain = info_gain(X, Y, i, Node.thresholds[i])
        if (bestGain[0] < gain[0]):
            bestGain = gain
            threshold = i
    node.threshold = threshold
    
    node.left = splitTree(Node(), bestGain[1], bestGain[3],dept+1)
    node.right = splitTree(Node(), bestGain[2],bestGain[4],dept+1)
    
    return node

In [152]:
root = Node(dept=0)
root.fillThreshold(X_train,Y_train,unique_threshold)

In [153]:
root = splitTree(root,X_train,Y_train,dept=0)

In [144]:
def printThresholds(node1):
    node = node1
    if node is not None: 
        print(node.threshold)
        printThresholds(node.left)
        printThresholds(node.right)
        print()

In [145]:
printThresholds(root)

7
22
None

21
28
None

1
25



27
None




20
1
15
None



13
4
3

None


1
15
None

16
8
4

None




None





